# Experiment: GGPN

**Note:** Using 50 dim due to memory constraints (limitation noted)

In [ ]:
# Setup
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import gc, json, warnings
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models.ggpn import GGPN
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

In [ ]:
# Training
print("Training GGPN (50 dim - memory constraint)...")

model = GGPN(train_data.num_entities, train_data.num_relations*2,
             embedding_dim=50, hidden_dim=50, num_layers=1, num_rff=20).to(device)
model.set_graph(train_data)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(50), desc="GGPN")):
    model.train()
    loss_sum, n = 0, 0
    for st in range(0, len(train_data), 512):
        pos = torch.tensor(train_data.triples[st:st+512], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()
        loss = model.loss(pos, neg)
        loss = loss['total'] if isinstance(loss, dict) else loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print("Training done!")

In [ ]:
# Evaluation
print("\nEvaluating...")
model.eval()

# MRR
ranks = []
with torch.no_grad():
    for i in tqdm(range(0, len(test_data), 200), desc="MRR", leave=False):
        batch = test_data.triples[i:i+200]
        h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
        all_e = torch.arange(train_data.num_entities, device=device)
        scores = torch.stack([model(h[j].expand(train_data.num_entities),
                                    r[j].expand(train_data.num_entities), all_e) for j in range(len(h))])
        target = scores[torch.arange(len(t), device=device), t]
        ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

ranks = torch.tensor(ranks, dtype=torch.float)
mrr = (1/ranks).mean().item()
h1 = (ranks <= 1).float().mean().item()
h10 = (ranks <= 10).float().mean().item()

# ECE
pos = test_data.triples
neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
all_t = np.vstack([pos, neg])
labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

confs = []
with torch.no_grad():
    for i in tqdm(range(0, len(all_t), 1024), desc="ECE", leave=False):
        batch = all_t[i:i+1024]
        h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
        scores = model(h, r, t)
        confs.append(torch.sigmoid(scores).cpu().numpy())
conf = np.concatenate(confs)
ece, _ = expected_calibration_error(conf, labels)
brier = brier_score(conf, labels)

# AUROC
ood_t = create_ood_dataset(train_data, test_data, "random", len(test_data))

def get_unc(triples):
    uncs = []
    with torch.no_grad():
        for i in range(0, len(triples), 1024):
            batch = triples[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            if hasattr(model, 'predict_with_uncertainty'):
                pred = model.predict_with_uncertainty(h, r, t)
                unc = pred.get('total', pred.get('epistemic'))
            else:
                s = model(h, r, t)
                p = torch.sigmoid(s)
                unc = -p * torch.log(p + 1e-10) - (1-p) * torch.log(1-p + 1e-10)
            uncs.append(unc.cpu().numpy())
    return np.concatenate(uncs)

auroc = compute_auroc(get_unc(test_data.triples), get_unc(ood_t))

results = {"mrr": mrr, "hits@1": h1, "hits@10": h10, "ece": ece, "brier": brier, "auroc": auroc}
print(f"\n{'='*50}")
print("GGPN RESULTS (50 dim)")
print(f"{'='*50}")
print(f"MRR: {mrr:.4f}")
print(f"Hits@1: {h1:.4f}")
print(f"Hits@10: {h10:.4f}")
print(f"ECE: {ece:.4f}")
print(f"Brier: {brier:.4f}")
print(f"AUROC: {auroc:.4f}")

In [ ]:
# Save
with open('ggpn_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved to ggpn_results.json")